# Machbarkeits-Check: Multi-Modal AI — Generative Images and VideoPrüft für jede der vier Kurs-Sessions, ob sie **(a)** über die OpenAI-API und **(b)** lokal auf dieserMaschine durchführbar ist. Jede Zelle schreibt ihr Ergebnis in `RESULTS`; die letzte Zelle baut darausdie Übersicht — dort steht also, was wirklich gelaufen ist, nicht was erwartet wurde.**Kernel:** Laeuft in jedem Kernel. Der Standard-Kernel (conda-Env `vllm`) hat kein `diffusers` —
die Setup-Zelle haengt dann das genvis-venv prozess-lokal in `sys.path`. Auf Platte aendert sich
dabei nichts: `vllm` behaelt `huggingface_hub` 1.21.0, nur der laufende Kernel-Prozess sieht 1.31.0.
Wer den Kernel `Python (genvis: diffusers)` waehlt, bekommt `diffusers` nativ — beides funktioniert.

**Kosten:** Alle lokalen Zellen sind gratis. Die Sora-Zelle kostet echtes Geld (~$0.40 pro 4-s-Clip)und ist deshalb hinter `RUN_PAID` gesperrt.

In [ ]:
from pathlib import Path
import os, sys, json, time, subprocess, importlib.util

# --- Kernel-Kompatibilitaet ------------------------------------------------
# Der vllm-Kernel hat kein diffusers. Dann wird das genvis-venv PROZESS-LOKAL
# in sys.path gehaengt: nichts auf Platte aendert sich, vllm bleibt bei
# huggingface_hub 1.21.0, nur dieser Kernel sieht 1.31.0.
GENVIS = "/home/ml/venvs/genvis/lib/python3.12/site-packages"
if importlib.util.find_spec("diffusers") is None:
    if not Path(GENVIS).is_dir():
        raise RuntimeError(
            f"Weder diffusers im Kernel noch genvis-venv unter {GENVIS}.\n"
            "Anlegen mit:\n"
            "  ~/miniconda3/envs/vllm/bin/python -m venv --system-site-packages /home/ml/venvs/genvis\n"
            "  /home/ml/venvs/genvis/bin/pip install diffusers")
    sys.path.insert(0, GENVIS)
    importlib.invalidate_caches()
    if importlib.util.find_spec("diffusers") is None:
        raise RuntimeError(f"sys.path-Injection fehlgeschlagen: {GENVIS} enthaelt kein diffusers.")
    print(f"diffusers aus genvis nachgeladen (Kernel selbst hat keins)")
else:
    print("diffusers nativ im Kernel vorhanden")

RESULTS = {}
def rec(session, check, ok, detail=""):
    RESULTS.setdefault(session, []).append({"check": check, "ok": ok, "detail": detail})
    print(f"{'OK  ' if ok else 'FAIL'} | {check}: {detail}")

OUT = Path("outputs_genvis"); OUT.mkdir(exist_ok=True)
RUN_PAID = False   # auf True setzen, um die Sora-Zelle wirklich auszufuehren

import torch, diffusers, huggingface_hub
print()
print("python      ", sys.executable)
print("torch       ", torch.__version__, "| cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu         ", torch.cuda.get_device_name(0),
          f"({torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB, sm_{''.join(map(str,torch.cuda.get_device_capability(0)))})")
print("diffusers   ", diffusers.__version__)
print("hf_hub      ", huggingface_hub.__version__)
print("ffmpeg      ", subprocess.run(["ffmpeg","-version"],capture_output=True,text=True).stdout.split()[2])


---
## Teil 1 — Was kann der OpenAI-Key?

Der Key liegt in `.env`. Zuerst: welche Modelle sind dem Projekt überhaupt freigeschaltet?

In [ ]:
from dotenv import load_dotenv
from openai import OpenAI, APIStatusError

load_dotenv("/home/ml/masterschool/.env")
client = OpenAI()

ids = sorted(m.id for m in client.models.list().data)
print(f"{len(ids)} Modelle freigeschaltet:")
for i in ids: print("  ", i)

visual = [i for i in ids if any(k in i for k in ("image","dall","sora","video"))]
rec("API", "Bild-/Video-Modelle in der Modellliste", bool(visual),
    ", ".join(visual) if visual else "keine — Liste enthaelt nur Text/Embedding")

In [ ]:
# Session 1 + 2 ueber die API: Bildgenerierung und -bearbeitung
for model in ("gpt-image-1", "dall-e-3"):
    try:
        client.images.generate(model=model, prompt="a red circle", n=1, size="1024x1024")
        rec("API", f"Bildgenerierung {model}", True, "funktioniert")
    except Exception as e:
        rec("API", f"Bildgenerierung {model}", False, str(e).split("-")[0][:90])

In [ ]:
# Vision-VERSTAENDNIS (nicht Generierung) — fuer die Abgrenzung in Session 1
import base64
from PIL import Image, ImageDraw

probe = OUT/"probe.png"
if not probe.exists():
    im = Image.new("RGB",(512,288),(20,24,40)); d = ImageDraw.Draw(im)
    d.ellipse([330,60,470,200], fill=(230,120,50))
    im.save(probe)

b64 = base64.b64encode(probe.read_bytes()).decode()
r = client.chat.completions.create(model="gpt-4o-mini", max_tokens=60, messages=[{
    "role":"user","content":[
        {"type":"text","text":"Describe this image in one short sentence."},
        {"type":"image_url","image_url":{"url":"data:image/png;base64,"+b64}}]}])
rec("API", "Vision-Verstaendnis (gpt-4o-mini)", True, r.choices[0].message.content.strip()[:80])

In [ ]:
# Session 3 ueber die API: Sora. KOSTET GELD — nur mit RUN_PAID=True.
if not RUN_PAID:
    print("uebersprungen (RUN_PAID=False). Setze RUN_PAID=True fuer einen echten ~$0.40-Lauf.")
else:
    import httpx
    key = os.environ["OPENAI_API_KEY"]
    with open(OUT/"hero_16x9.png" if (OUT/"hero_16x9.png").exists() else probe,"rb") as fh:
        files = {"input_reference": ("ref.png", fh, "image/png")}
        data  = {"model":"sora-2","prompt":"slow cinematic zoom in, stable composition",
                 "seconds":"4","size":"1280x720"}
        job = httpx.post("https://api.openai.com/v1/videos",
                         headers={"Authorization":f"Bearer {key}"},
                         data=data, files=files, timeout=60).json()
    vid = job["id"]; print("job:", vid, job["status"])
    while job["status"] in ("queued","in_progress"):
        time.sleep(10)
        job = httpx.get(f"https://api.openai.com/v1/videos/{vid}",
                        headers={"Authorization":f"Bearer {key}"}, timeout=60).json()
        print(" ", job["status"], job.get("progress"))
    if job["status"] == "completed":
        mp4 = httpx.get(f"https://api.openai.com/v1/videos/{vid}/content",
                        headers={"Authorization":f"Bearer {key}"}, timeout=300).content
        (OUT/"api_sora.mp4").write_bytes(mp4)
        rec("API", "Video sora-2 (image-to-video)", True, f"{len(mp4)/1e6:.1f} MB")
    else:
        rec("API", "Video sora-2 (image-to-video)", False, str(job.get("error"))[:80])

---
## Teil 2 — Session 1 lokal: Text-to-Image

SDXL-base. Erster Lauf lädt ~7 GB in den HF-Cache, danach kommt das Modell von Platte.
Der Prompt ist wörtlich der aus dem Kursentwurf.

In [ ]:
import torch, gc
from diffusers import StableDiffusionXLPipeline

def free(*objs):
    for o in objs:
        try: del o
        except Exception: pass
    gc.collect(); torch.cuda.empty_cache()

HERO_PROMPT = ("Create a clean 16:9 hero image for a course called Multi-Modal AI: Generative "
               "Images and Video. Show abstract connections between images, video frames, and "
               "neural network patterns. Modern educational style, minimal composition, soft "
               "lighting, empty space on the left for title text.")
NEG = "text, letters, watermark, logo, human face, people, clutter, busy background"

t0 = time.time()
pipe = StableDiffusionXLPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    torch_dtype=torch.float16, variant="fp16", use_safetensors=True).to("cuda")
load_s = time.time()-t0

t1 = time.time()
hero = pipe(prompt=HERO_PROMPT, negative_prompt=NEG, width=1344, height=768,
            num_inference_steps=30, guidance_scale=6.0,
            generator=torch.Generator("cuda").manual_seed(42)).images[0]
gen_s = time.time()-t1
hero.save(OUT/"hero_16x9.png")
rec("Session 1", "SDXL text-to-image", True,
    f"{gen_s:.1f}s/Bild (laden {load_s:.0f}s), {torch.cuda.max_memory_allocated()/1e9:.1f} GB peak")
free(pipe)
hero

---
## Session 2 lokal: Bild bearbeiten

Zwei verschiedene Ziele aus dem Kursentwurf, die **unterschiedliche Verfahren** brauchen:

| Ziel | Verfahren | Zusatz-Download |
|---|---|---|
| Stil, Palette, Aufräumen | img2img | keiner (gleiche SDXL-Gewichte) |
| Freie Fläche für den Titel | Outpainting | Inpaint-Checkpoint, 6.9 GB |

Das ist der Punkt, an dem der Kursentwurf ungenau ist: img2img ändert den *Stil*, nicht das *Layout*.

In [ ]:
from diffusers import StableDiffusionXLImg2ImgPipeline
from PIL import Image

BANNER_PROMPT = ("clean modern 16:9 course banner, abstract generative AI and video theme, "
                 "simplified uncluttered background, deep blue and warm orange palette, "
                 "large empty negative space on the left side, professional educational style")

pipe = StableDiffusionXLImg2ImgPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    torch_dtype=torch.float16, variant="fp16", use_safetensors=True).to("cuda")

t1 = time.time()
banner = pipe(prompt=BANNER_PROMPT, negative_prompt=NEG,
              image=Image.open(OUT/"hero_16x9.png").convert("RGB"),
              strength=0.45, guidance_scale=7.0, num_inference_steps=35,
              generator=torch.Generator("cuda").manual_seed(7)).images[0]
banner.save(OUT/"banner_img2img.png")
rec("Session 2", "img2img (Stil/Palette)", True, f"{time.time()-t1:.1f}s")
free(pipe)
banner

In [ ]:
from diffusers import StableDiffusionXLInpaintPipeline

pipe = StableDiffusionXLInpaintPipeline.from_pretrained(
    "diffusers/stable-diffusion-xl-1.0-inpainting-0.1",
    torch_dtype=torch.float16, variant="fp16").to("cuda")

src = Image.open(OUT/"banner_img2img.png").convert("RGB").resize((1024,576))
W,H = 1024,576; shift = int(W*0.40)
canvas = Image.new("RGB",(W,H),(245,245,248)); canvas.paste(src.resize((W-shift,H)),(shift,0))
mask = Image.new("L",(W,H),0); mask.paste(255,(0,0,shift+24,H))

t1 = time.time()
out = pipe(prompt="soft empty gradient background, clean minimal negative space, "
                  "subtle pale blue tone, no objects, no lines, no text",
           negative_prompt="lines, network, nodes, objects, text, clutter, detail",
           image=canvas, mask_image=mask, width=W, height=H,
           strength=0.99, guidance_scale=7.5, num_inference_steps=35,
           generator=torch.Generator("cuda").manual_seed(11)).images[0]
out.save(OUT/"banner_outpaint.png")
rec("Session 2", "Outpainting (Titelflaeche)", True,
    f"{time.time()-t1:.1f}s - aber mit sichtbarer Naht, siehe unten")
free(pipe)
out


### Die Naht — und was wirklich dagegen hilft

Das Bild oben hat eine harte Kante bei 40 %. Zwei Dinge, die **nicht** helfen (beide getestet):

- **Weichgezeichnete Maske.** `StableDiffusionXLInpaintPipeline` binarisiert die Maske bei 0.5,
  der Gaussian-Blur wird verworfen.
- **Groessere Maskenueberlappung** (160 px, 300 px). Die Naht bleibt exakt bei 40 %, zusaetzlich
  entstehen vertikale Streifenartefakte. Ursache: das Canvas hat an der Klebekante einen harten
  Helligkeitssprung, den der VAE mitkodiert.

Der Titelbereich ist kein Generierungs-, sondern ein **Kompositionsproblem**. Ein Verlaufs-Scrim
loest es deterministisch, in unter einer Sekunde, ohne GPU — so arbeiten Designer auch.


In [ ]:
# Scrim: Hintergrundfarbe links deckend, nach rechts nicht-linear auslaufend.
t1 = time.time()
src = Image.open(OUT/"banner_img2img.png").convert("RGB").resize((1344,768))
W,H = src.size
bg = Image.new("RGB",(W,H),(247,248,251))
grad = Image.new("L",(W,H)); px = grad.load()
x0, x1 = int(W*0.02), int(W*0.58)     # voll deckend bis 2%, transparent ab 58%
for x in range(W):
    a = 255 if x <= x0 else 0 if x >= x1 else int(255 * (1-(x-x0)/(x1-x0))**2.2)
    for y in range(H): px[x,y] = a
banner_final = Image.composite(bg, src, grad)
banner_final.save(OUT/"banner_scrim.png")
rec("Session 2", "Verlaufs-Scrim (Titelflaeche, nahtlos)", True,
    f"{time.time()-t1:.2f}s, keine GPU")
banner_final


---
## Session 3 lokal: Image-to-Video

Stable Video Diffusion XT, fp16-Variante (~4.5 GB). Läuft mit `enable_model_cpu_offload()` in 16 GB VRAM.

**Wichtige Einschränkung:** SVD kennt **keine Motion-Prompts**. Bewegung wird über `motion_bucket_id`
(niedrig = ruhig, hoch = viel Bewegung) gesteuert, nicht über Sprache. Das Kapitel "Motion Prompting"
aus dem Kursentwurf — *slow zoom-in*, *gentle parallax* — funktioniert so lokal nicht.

In [ ]:
from diffusers import StableVideoDiffusionPipeline

pipe = StableVideoDiffusionPipeline.from_pretrained(
    "stabilityai/stable-video-diffusion-img2vid-xt",
    torch_dtype=torch.float16, variant="fp16")
pipe.enable_model_cpu_offload()

img = Image.open(OUT/"banner_scrim.png").convert("RGB").resize((1024,576))
t1 = time.time()
frames = pipe(img, decode_chunk_size=8, num_frames=25,
              motion_bucket_id=90, noise_aug_strength=0.02,
              generator=torch.Generator("cuda").manual_seed(3)).frames[0]
gen_s = time.time()-t1

frdir = OUT/"frames"; frdir.mkdir(exist_ok=True)
for i,f in enumerate(frames): f.save(frdir/f"{i:03d}.png")
subprocess.run(["ffmpeg","-y","-loglevel","error","-framerate","8","-i",str(frdir/"%03d.png"),
                "-vf","scale=1024:576,format=yuv420p","-r","24",str(OUT/"local_svd.mp4")], check=True)
rec("Session 3", "SVD image-to-video (lokal)", True,
    f"{len(frames)} frames in {gen_s:.0f}s, {torch.cuda.max_memory_allocated()/1e9:.1f} GB peak")
free(pipe)

from IPython.display import Video
Video(str(OUT/"local_svd.mp4"), embed=True, width=640)

In [ ]:
# Der Fallback aus dem Kursentwurf: Ken-Burns-Zoom, rein ffmpeg, kein Modell.
t1 = time.time()
subprocess.run(["ffmpeg","-y","-loglevel","error","-loop","1","-i",str(OUT/"banner_scrim.png"),
                "-vf","zoompan=z='min(zoom+0.0015,1.25)':d=150:x='iw/2-(iw/zoom/2)':"
                      "y='ih/2-(ih/zoom/2)':s=1280x720,format=yuv420p",
                "-t","5","-r","30",str(OUT/"kenburns.mp4")], check=True)
rec("Session 3", "ffmpeg Ken-Burns-Fallback", True, f"{time.time()-t1:.1f}s, kein Modell noetig")

---
## Session 4 lokal: Brief → Asset-Kit

Setzt nur zusammen, was die Sessions 1-3 erzeugt haben. Keine weiteren Modelle.

In [ ]:
import datetime

brief = {
  "course":"Multi-Modal AI: Generative Images and Video",
  "audience":"beginner to intermediate AI learners",
  "style":"modern, clean, educational",
  "constraints":["no readable text","no logos","no human faces"],
}

hero = Image.open(OUT/"banner_scrim.png").convert("RGB")
hero.resize((1920,1080), Image.LANCZOS).save(OUT/"kit_hero_16x9.png")

w,h = hero.size
hero.crop((w-h,0,w,h)).resize((1024,1024), Image.LANCZOS).save(OUT/"kit_thumbnail_1x1.png")

src = OUT/"local_svd.mp4" if (OUT/"local_svd.mp4").exists() else OUT/"kenburns.mp4"
subprocess.run(["ffmpeg","-y","-loglevel","error","-i",str(src),
                "-vf","scale=1920:1080,format=yuv420p","-r","24",
                str(OUT/"kit_teaser.mp4")], check=True)

json.dump({"brief":brief, "video_source":src.name,
           "generated_at":datetime.date.today().isoformat(),
           "prompts":{"hero":HERO_PROMPT,"banner":BANNER_PROMPT,"negative":NEG},
           "models":{"image":"stabilityai/stable-diffusion-xl-base-1.0",
                     "edit":"diffusers/stable-diffusion-xl-1.0-inpainting-0.1",
                     "video":"stabilityai/stable-video-diffusion-img2vid-xt"}},
          open(OUT/"kit_prompts.json","w"), indent=2)

kit = sorted(p.name for p in OUT.glob("kit_*"))
rec("Session 4", "Kit-Assemblierung (Brief -> Assets)", len(kit)==4, ", ".join(kit))

---
## Ergebnis

In [ ]:
print(f"{'Session':<11} {'Check':<42} {'':4} Detail")
print("-"*110)
for sess, checks in RESULTS.items():
    for c in checks:
        print(f"{sess:<11} {c['check']:<42} {'OK' if c['ok'] else 'FAIL':<4} {c['detail'][:48]}")

api_visual = any(c["ok"] for c in RESULTS.get("API",[]) if "Bild" in c["check"])
local_ok   = all(c["ok"] for s in ("Session 1","Session 2","Session 3","Session 4")
                 for c in RESULTS.get(s,[]))
print("\n" + "="*110)
print(f"Bildgenerierung ueber die API : {'ja' if api_visual else 'NEIN — Projekt hat keine Bildmodelle'}")
print(f"Alle vier Sessions lokal      : {'ja' if local_ok else 'nein, siehe FAIL oben'}")